# project_3_pr_dm.ipynb

- 목적: 재난 문자 원본을 1차 정리하고, 지진/해일 대피소 원본을 앱에서 쓰기 쉬운 기본 컬럼으로 바꾼다.
- 입력: 재난 문자 원본, 지진 대피소 원본, 해일 대피소 원본
- 출력: `disaster_message_clean.csv`, `earthquake_shelter_clean.csv`, `tsunami_shelter_clean.csv`
- 메모: 하나의 노트북 안에 재난 문자 정리와 두 종류 대피소 전처리 단계가 함께 들어 있다.


## 1. 재난 문자 원본 1차 정리
재난 문자 원본에서 실종/기타 메시지를 걷어내고 후속 분석용 정리본을 저장한다.


In [ ]:
# 재난 문자 원본에서 추천과 직접 관련 없는 메시지를 제거한다.
# 지진 대피소 원본을 확인하고 필요한 컬럼만 남길 준비를 한다.
import pandas as pd

df = pd.read_csv("data/disaster_clean_2020_2024.csv")

# 실종 관련 문자 제거
df = df[~df["MSG_CN"].str.contains("찾습니다|실종|배회|목격", na=False)]

# 재난 유형 제거
df = df[~df["DST_SE_NM"].str.contains("기타|교통통제|가축질병", na=False)]

print("전처리 후 데이터 수:", len(df))

df.to_csv("data/disaster_message_clean.csv", index=False, encoding="utf-8-sig")


## 2. 지진 대피소 원본 정리
원본 지진 대피소 CSV에서 필요한 컬럼만 골라 한글 컬럼명으로 바꾼다.


In [ ]:
# 지진 대피소 원본을 확인하고 필요한 컬럼만 남길 준비를 한다.
df = pd.read_csv("data/earthquake_shelter_raw.csv")

df.head()


In [ ]:
df.columns


In [ ]:
df = df[
    [
        "VT_ACMDFCLTY_NM",   # 대피소 이름
        "RN_DTL_ADRES",      # 도로명 주소
        "LA",                # 위도
        "LO",                # 경도
        "VT_ACMD_PSBL_NMPR"  # 수용 가능 인원
    ]
]

df.head()


In [ ]:
df = df.rename(columns={
    "VT_ACMDFCLTY_NM": "대피소명",
    "RN_DTL_ADRES": "주소",
    "LA": "위도",
    "LO": "경도",
    "VT_ACMD_PSBL_NMPR": "수용인원"
})


In [ ]:
df = df.dropna()


In [ ]:
df.info()
df.head()


In [ ]:
# 해일 대피소 원본도 같은 방식으로 읽어 후속 정리를 시작한다.
df.to_csv(
    "data/earthquake_shelter_clean.csv",
    index=False,
    encoding="utf-8-sig"
)


## 3. 지진 대피소 위치 확인
무료 지도 위에 좌표를 올려 지진 대피소 데이터가 정상적으로 들어왔는지 확인한다.


In [ ]:
import plotly.express as px

fig = px.scatter_mapbox(
    df,
    lat="위도",
    lon="경도",
    hover_name="대피소명",
    hover_data=["주소", "수용인원"],
    zoom=6,
    height=700,
    title="대한민국 지진 대피소 지도"
)

fig.update_layout(
    mapbox_style="open-street-map"
)

fig.show()


## 4. 해일 대피소 원본 정리
같은 방식으로 해일 대피소 원본을 선택/변환해 후속 권역 필터링 준비를 한다.


In [ ]:
# 해일 대피소 원본도 같은 방식으로 읽어 후속 정리를 시작한다.
import plotly.express as px

fig = px.scatter_mapbox(
    df,
    lat="위도",
    lon="경도",
    hover_name="대피소명",
    hover_data=["주소", "수용인원"],
    zoom=6,
    height=700,
    title="대한민국 지진 대피소 지도"
)

fig.update_layout(
    mapbox_style="open-street-map"
)

fig.show()


In [ ]:
import pandas as pd

df = pd.read_csv("data/tsunami_shelter_raw.csv")

df.head()


In [ ]:
df.columns


In [ ]:
df = df[
    [
        "SHNT_PLACE_NM",
        "RN_DTL_ADRES",
        "LA",
        "LO",
        "PSBL_NMPR"
    ]
]

df.head()


In [ ]:
df = df.rename(columns={
    "SHNT_PLACE_NM": "대피소명",
    "RN_DTL_ADRES": "주소",
    "LA": "위도",
    "LO": "경도",
    "PSBL_NMPR": "수용인원"
})


In [ ]:
df = df.dropna()


In [ ]:
df["위도"] = pd.to_numeric(df["위도"])
df["경도"] = pd.to_numeric(df["경도"])


In [ ]:
df["지역"] = df["주소"].str.split().str[0]


## 5. 해일 대피소 위치 확인
좌표와 분포를 지도에서 확인해 전처리 결과를 빠르게 검토한다.


In [ ]:
fig = px.scatter_mapbox(
    df,
    lat="위도",
    lon="경도",
    hover_name="대피소명",
    hover_data=["주소", "수용인원"],
    zoom=6,
    height=700,
    title="지진해일 대피소 지도"
)

fig.update_layout(mapbox_style="open-street-map")

fig.show()
